In [1]:
# input
blast_result = "./tmp/human_258_to_11_species.blast"
fasta_file = "./tmp/entryId-repId-taxId.fasta"
tax_file = "./tmp/entryId-repId-taxId.tsv"
# output
clustalo_dir = "./tmp/clustalo/"


In [2]:
import pandas as pd
from Bio import SeqIO
import os

id2seq_record = dict()
for r in SeqIO.parse(fasta_file, "fasta"):
    id2seq_record[r.id] = r

df = pd.read_table(blast_result, header=None)
df.rename(columns={0: "seq_id", 1: "target_seq_id", 10: "evalue", 11: "bitscore"}, inplace=True)
df_tax = pd.read_table(tax_file, header=None, names=["target_seq_id", "rep_id", "tax_id"])
df = pd.merge(df, df_tax, on="target_seq_id", how="left")

len(df)

18859

In [3]:
for (seq_id_af2, ), df_seq in df.groupby(by=['seq_id']):
    df_seq = df_seq.sort_values(by=['bitscore'], ascending=False).drop_duplicates(['tax_id'], keep="first")

    seq_id_af2: str
    seq_id = seq_id_af2.split("-")[1]
    out_dir = f"{clustalo_dir}/{seq_id}"
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)

    records = [id2seq_record[i] for i in df_seq['target_seq_id'].tolist()]
    out_file = f"{out_dir}/{seq_id}.fasta"
    _ = SeqIO.write(records, out_file, "fasta")